In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# Load cleaned data
df = pd.read_csv("../data/processed/loans_clean.csv")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Clean int_rate — remove % sign if present
df["int_rate"] = (df["int_rate"]
                  .astype(str)
                  .str.replace("%", "", regex=False)
                  .astype(float))

# Clean term — extract number only (e.g. "36 months" → 36)
df["term"] = (df["term"]
              .astype(str)
              .str.extract(r"(\d+)")[0]
              .astype(float))

# Clean emp_length — extract number only (e.g. "10+ years" → 10)
df["emp_length"] = (df["emp_length"]
                    .astype(str)
                    .str.extract(r"(\d+)")[0]
                    .astype(float))

# Clean revol_util — remove % sign if present
df["revol_util"] = (df["revol_util"]
                    .astype(str)
                    .str.replace("%", "", regex=False)
                    .astype(float))

# Fill missing values with median for numeric columns
for col in ["annual_inc", "dti", "revol_util", "emp_length"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: filled {df[col].isna().sum()} missing → median {median_val:.2f}")

print(f"\nCleaning complete")
print(df[["int_rate","term","emp_length","revol_util"]].describe().round(2))

Loaded: 1,386,663 rows, 18 columns
  annual_inc: filled 0 missing → median 65000.00
  dti: filled 0 missing → median 17.61
  revol_util: filled 0 missing → median 51.80
  emp_length: filled 0 missing → median 6.00

Cleaning complete
         int_rate        term  emp_length  revol_util
count  1386663.00  1386663.00  1386663.00  1386663.00
mean        13.25       41.80        6.04       51.43
std          4.78       10.28        3.46       24.64
min          5.31       36.00        1.00        0.00
25%          9.75       36.00        3.00       32.90
50%         12.74       36.00        6.00       51.80
75%         16.02       36.00       10.00       70.40
max         30.99       60.00       10.00      892.30


In [3]:
# Cap revol_util at 100 — anything above is a data error
print(f"revol_util above 100%: {(df['revol_util'] > 100).sum():,} rows")
df["revol_util"] = df["revol_util"].clip(upper=100)

# Also cap annual_inc outliers — top 1% are extreme outliers
p99 = df["annual_inc"].quantile(0.99)
print(f"annual_inc 99th percentile: ${p99:,.0f}")
df["annual_inc"] = df["annual_inc"].clip(upper=p99)

# Cap dti — anything above 60 is unreliable
print(f"dti above 60: {(df['dti'] > 60).sum():,} rows")
df["dti"] = df["dti"].clip(upper=60)

# Verify fixes
print("\nAfter capping:")
print(df[["int_rate","term","emp_length",
          "revol_util","annual_inc","dti"]].describe().round(2))

revol_util above 100%: 4,783 rows
annual_inc 99th percentile: $255,000
dti above 60: 2,098 rows

After capping:
         int_rate        term  emp_length  revol_util  annual_inc         dti
count  1386663.00  1386663.00  1386663.00  1386663.00   1386663.0  1386663.00
mean        13.25       41.80        6.04       51.42     74840.9       18.20
std          4.78       10.28        3.46       24.61     42578.2        8.70
min          5.31       36.00        1.00        0.00         0.0       -1.00
25%          9.75       36.00        3.00       32.90     46000.0       11.77
50%         12.74       36.00        6.00       51.80     65000.0       17.61
75%         16.02       36.00       10.00       70.40     90477.0       24.07
max         30.99       60.00       10.00      100.00    255000.0       60.00


In [4]:
# Define features for the scorecard
feature_cols = [
    "loan_amnt", "term", "int_rate", "emp_length",
    "annual_inc", "dti", "delinq_2yrs", "open_acc",
    "revol_bal", "revol_util", "total_acc"
]

X = df[feature_cols]
y = df["target"]

# 80/20 stratified split — preserves default rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size:         {X_train.shape[0]:,} rows")
print(f"Test size:          {X_test.shape[0]:,} rows")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate:  {y_test.mean():.2%}")
print(f"\nFeatures: {feature_cols}")

Train size:         1,109,330 rows
Test size:          277,333 rows
Train default rate: 19.82%
Test default rate:  19.82%

Features: ['loan_amnt', 'term', 'int_rate', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'open_acc', 'revol_bal', 'revol_util', 'total_acc']


In [5]:
# Information Value — Basel standard feature selector
def calc_iv(df, feature, target, bins=10):
    """
    Calculate Information Value for a numeric feature.
    IV > 0.02 = useful, IV > 0.1 = strong, IV > 0.3 = very strong
    """
    df2 = df[[feature, target]].copy()
    
    # Bin the feature into quantiles
    df2["bin"] = pd.qcut(df2[feature], q=bins, duplicates="drop")
    
    # Count events (defaults) and non-events (good loans) per bin
    grouped = df2.groupby("bin", observed=True)[target].agg(
        events=lambda x: (x == 1).sum(),
        non_events=lambda x: (x == 0).sum()
    ).reset_index()
    
    # Calculate distributions
    total_events = grouped["events"].sum()
    total_non    = grouped["non_events"].sum()
    
    grouped["dist_events"] = grouped["events"] / total_events
    grouped["dist_non"]    = grouped["non_events"] / total_non
    
    # Weight of Evidence and Information Value
    grouped["woe"] = np.log(
        (grouped["dist_non"]    + 1e-4) /
        (grouped["dist_events"] + 1e-4)
    )
    grouped["iv"] = (grouped["dist_non"] - grouped["dist_events"]) * grouped["woe"]
    
    return grouped["iv"].sum()

# Build training dataframe for IV calculation
train_df = X_train.copy()
train_df["target"] = y_train.values

# Score every feature
iv_scores = {}
for col in feature_cols:
    try:
        iv_scores[col] = round(calc_iv(train_df, col, "target"), 4)
    except Exception as e:
        iv_scores[col] = 0
        print(f"  Could not calculate IV for {col}: {e}")

# Display results with predictive power labels
iv_df = (pd.DataFrame.from_dict(iv_scores, orient="index", columns=["IV"])
           .sort_values("IV", ascending=False))

iv_df["Predictive Power"] = pd.cut(
    iv_df["IV"],
    bins=[-1, 0.02, 0.1, 0.3, 0.5, 99],
    labels=["Useless", "Weak", "Medium", "Strong", "Very Strong"]
)

print("=" * 45)
print("   INFORMATION VALUE — FEATURE RANKING")
print("=" * 45)
print(iv_df.to_string())
print("=" * 45)
print("\nFeatures to KEEP (IV > 0.02):")
keep = iv_df[iv_df["IV"] > 0.02].index.tolist()
print(keep)
print(f"\nFeatures to DROP (IV ≤ 0.02):")
drop = iv_df[iv_df["IV"] <= 0.02].index.tolist()
print(drop)

   INFORMATION VALUE — FEATURE RANKING
                 IV Predictive Power
int_rate     0.4443           Strong
dti          0.0703             Weak
loan_amnt    0.0342             Weak
annual_inc   0.0290             Weak
revol_util   0.0280             Weak
emp_length   0.0067          Useless
open_acc     0.0045          Useless
revol_bal    0.0035          Useless
total_acc    0.0018          Useless
delinq_2yrs  0.0017          Useless
term         0.0000          Useless

Features to KEEP (IV > 0.02):
['int_rate', 'dti', 'loan_amnt', 'annual_inc', 'revol_util']

Features to DROP (IV ≤ 0.02):
['emp_length', 'open_acc', 'revol_bal', 'total_acc', 'delinq_2yrs', 'term']


In [6]:
# Final feature list based on IV > 0.02
selected_features = ['int_rate', 'dti', 'loan_amnt', 'annual_inc', 'revol_util']

print("Selected features:")
for f in selected_features:
    print(f"  {f:15s}  IV: {iv_scores[f]:.4f}  "
          f"Power: {iv_df.loc[f,'Predictive Power']}")

# Build final train and test sets
X_train_final = X_train[selected_features].copy()
X_test_final  = X_test[selected_features].copy()

# Save to processed folder
train_out = X_train_final.copy()
train_out["target"] = y_train.values
train_out.to_csv("../data/processed/train.csv", index=False)

test_out = X_test_final.copy()
test_out["target"] = y_test.values
test_out.to_csv("../data/processed/test.csv", index=False)

print(f"\nTrain saved: {train_out.shape[0]:,} rows × {train_out.shape[1]} columns")
print(f"Test saved:  {test_out.shape[0]:,} rows × {test_out.shape[1]} columns")
print(f"\nFeature engineering complete — move to 03_modelling.ipynb")

Selected features:
  int_rate         IV: 0.4443  Power: Strong
  dti              IV: 0.0703  Power: Weak
  loan_amnt        IV: 0.0342  Power: Weak
  annual_inc       IV: 0.0290  Power: Weak
  revol_util       IV: 0.0280  Power: Weak

Train saved: 1,109,330 rows × 6 columns
Test saved:  277,333 rows × 6 columns

Feature engineering complete — move to 03_modelling.ipynb
